## Prepare bølgemodel data

In [7]:
import geopandas as gpd
import waves

from pathlib import Path


In [11]:
waves.prepare.bolge_model_data()

Rasterizing aoi mask...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:49.
AOI mask ready: /home/kim/work/eswm-bolgemodell/niva/tmp_outline_mask.tif
Copying raster for filling...
Filling nodata (0) inside outline...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:28:48.
Coarse fill: downsampling 20× ...
Coarse fill: FillNodata at low resolution...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:12.
Coarse fill: upsampling back to original resolution...
Coarse fill: merging into raster (remaining nodata only)...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:01:23.
Clipping raster to outline...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:44.
Filled COG saved: /home/kim/work/eswm-bolgemodell/niva/EswmRaster_filled_cog.tif
COG saved: /home/kim/work/eswm-bolgemodell/niva/EswmRaster_clipped_cog.tif


## NiN 3 – bølgeeksponering (SWM-klassifisering)

Mapping of SWM (Shore Wave Model) raster values to NiN 3 wave-exposure classes.

| Klasse | Trinn | Trinn-navn (NO) | English name | SWM-grenseverdier |
|--------|-------|-----------------|--------------|-------------------|
| 1 | 0 | minimal vannforstyrrelsesintensitet | still water | < 1 200 |
| 2 | a | svært beskyttet | very sheltered | ≥ 1 200, < 4 000 |
| 3 | b | temmelig beskyttet | moderately sheltered | ≥ 4 000, < 10 000 |
| 4 | c | litt beskyttet | slightly sheltered | ≥ 10 000, < 50 000 |
| 5 | d | svakt eksponert | weakly sheltered | ≥ 50 000, < 100 000 |
| 6 | e | litt eksponert | slightly exposed | ≥ 100 000, < 500 000 |
| 7 | f | temmelig eksponert | moderately exposed | ≥ 500 000, < 1 000 000 |
| 8 | g | svært eksponert | very exposed | ≥ 1 000 000, < 2 000 000 |
| 9 | h | ekstremt eksponert | extremely exposed | ≥ 2 000 000, < 4 000 000 |
| 10 | y | disruptivt eksponert | disruptively exposed | ≥ 4 000 000 |


In [2]:
waves.classify.CLASSES

,class_int,trinn,navn_no,navn_en,swm_lower,swm_upper
0,1,0,minimal vannforstyrrelsesintensitet,still water,NaN,1200.0
1,2,a,svært beskyttet,very sheltered,1200.0,4000.0
2,3,b,temmelig beskyttet,moderately sheltered,4000.0,10000.0
3,4,c,litt beskyttet,slightly sheltered,10000.0,50000.0
4,5,d,svakt eksponert,weakly sheltered,50000.0,100000.0
5,6,e,litt eksponert,slightly exposed,100000.0,500000.0
6,7,f,temmelig eksponert,moderately exposed,500000.0,1000000.0
7,8,g,svært eksponert,very exposed,1000000.0,2000000.0
8,9,h,ekstremt eksponert,extremely exposed,2000000.0,4000000.0
9,10,y,disruptivt eksponert,disruptively exposed,4000000.0,NaN


In [16]:
waves.classify.reclassify_raster(SRC, CLF)
print(f"Classified raster saved: {CLF}")

0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:48.
Classified raster saved: ../niva/EswmRaster_classified.tif


In [17]:
waves.classify.sieve_filter(CLF, SIE, threshold=4)
print(f"Sieved raster saved: {SIE}")

Copying classified raster...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:12.
Applying sieve filter (threshold=4, connectedness=8)...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:01:15.
Sieved raster saved: ../niva/EswmRaster_classified_sieved.tif


In [18]:
waves.classify.polygonize(SIE, VRAW)
print(f"Raw polygons saved: {VRAW}")

Polygonizing (this may take a while)...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:01:09.
Raw polygons saved: ../niva/Eswm_classified_raw.gpkg


In [8]:
ROOT_PATH = Path("..")
gdf = gpd.read_file(ROOT_PATH / "niva/Eswm_land_subtract_checkpoint.gpkg")
gdf.head()

,class_int,geometry
0,8,"MULTIPOLYGON (((1100391.631 7945231.561, 11003..."
1,8,"MULTIPOLYGON (((1105066.631 7941331.561, 11050..."
2,9,"MULTIPOLYGON (((1100041.631 7980831.561, 11000..."
3,7,"MULTIPOLYGON (((953841.631 7939956.561, 953841..."
4,8,"MULTIPOLYGON (((953841.631 7939931.561, 953841..."


In [9]:
crs = "EPSG:25833"

land = gpd.read_file("land_union-from-NDH-DTM-Norge.gpkg")#.to_crs(crs)

In [ ]:

#12754 ->12000

from shapely.ops import unary_union
import numpy as np
from shapely.geometry import box

AREA_THRESHOLD = 37_611_003

def split_geometry_grid(geom, n=10):
    """Split geometry into n x n grid cells."""
    minx, miny, maxx, maxy = geom.bounds
    xs = np.linspace(minx, maxx, n + 1)
    ys = np.linspace(miny, maxy, n + 1)
    pieces = []
    for i in range(n):
        for j in range(n):
            print(f"  Creating grid cell {i * n + j + 1}/{n * n}")
            cell = box(xs[i], ys[j], xs[i + 1], ys[j + 1])
            piece = geom.intersection(cell)
            if not piece.is_empty:
                pieces.append(piece)
    return pieces

CHECKPOINT_PATH = ROOT_PATH / "niva/Eswm_land_subtract_checkpoint.gpkg"

n = len(land.geometry)
for i, land_geom in enumerate(land.geometry[11999:], start=12000):
    if i ==  12754:
        break
    print(f"Subtracting land geometry {i} of {n}")
    if land_geom.area > AREA_THRESHOLD:
        pieces = split_geometry_grid(land_geom, n=10)
        for j, piece in enumerate(pieces, 1):
            print(f"  Processing piece {j}/{len(pieces)} of {land_geom.area/10000:.2f} ha")
            mask = gdf.sindex.query(piece, predicate="intersects")
            if len(mask) > 0:
                gdf.geometry.iloc[mask] = gdf.geometry.iloc[mask].difference(piece)
            if j % 1000 == 0:
                print(f"  Checkpoint at {j} – saving to {CHECKPOINT_PATH}")
                gdf[~gdf.is_empty].to_file(CHECKPOINT_PATH, driver="GPKG")
    else:
        mask = gdf.sindex.query(land_geom, predicate="intersects")
        if len(mask) > 0:
            gdf.geometry.iloc[mask] = gdf.geometry.iloc[mask].difference(land_geom)

    if i % 1000 == 0:
        print(f"  Checkpoint at {i} – saving to {CHECKPOINT_PATH}")
        gdf[~gdf.is_empty].to_file(CHECKPOINT_PATH, driver="GPKG")

gdf = gdf[~gdf.is_empty].reset_index(drop=True)
gdf.to_file(CHECKPOINT_PATH, driver="GPKG")
print(f"Final checkpoint saved: {CHECKPOINT_PATH}")
gdf.head()
152999


Subtracting land geometry 12000 of 328292
  Checkpoint at 12000 – saving to ../niva/Eswm_land_subtract_checkpoint.gpkg
Subtracting land geometry 12001 of 328292
Subtracting land geometry 12002 of 328292
Subtracting land geometry 12003 of 328292
Subtracting land geometry 12004 of 328292
Subtracting land geometry 12005 of 328292
Subtracting land geometry 12006 of 328292
Subtracting land geometry 12007 of 328292
Subtracting land geometry 12008 of 328292
Subtracting land geometry 12009 of 328292
Subtracting land geometry 12010 of 328292
Subtracting land geometry 12011 of 328292
Subtracting land geometry 12012 of 328292
Subtracting land geometry 12013 of 328292
Subtracting land geometry 12014 of 328292
Subtracting land geometry 12015 of 328292
Subtracting land geometry 12016 of 328292
Subtracting land geometry 12017 of 328292
Subtracting land geometry 12018 of 328292
Subtracting land geometry 12019 of 328292
Subtracting land geometry 12020 of 328292
Subtracting land geometry 12021 of 328292

,class_int,geometry
0,8,"MULTIPOLYGON (((1100391.631 7945231.561, 11003..."
1,8,"MULTIPOLYGON (((1105066.631 7941331.561, 11050..."
2,9,"MULTIPOLYGON (((1100041.631 7980831.561, 11000..."
3,7,"MULTIPOLYGON (((953841.631 7939956.561, 953841..."
4,8,"MULTIPOLYGON (((953841.631 7939931.561, 953841..."


In [10]:
cols = ["class_int", "trinn", "navn_no", "navn_en"]
gdf = gdf.merge(waves.classify.CLASSES[cols], on="class_int", how="left")
gdf.head()

,class_int,geometry,trinn,navn_no,navn_en
0,7,"POLYGON ((953841.631 7939956.561, 953841.631 7...",f,temmelig eksponert,moderately exposed
1,6,"POLYGON ((953841.631 7939931.561, 953841.631 7...",e,litt eksponert,slightly exposed
2,7,"POLYGON ((953541.631 7939806.561, 953541.631 7...",f,temmelig eksponert,moderately exposed
3,7,"POLYGON ((973891.631 7939706.561, 973891.631 7...",f,temmelig eksponert,moderately exposed
4,7,"POLYGON ((973791.631 7939881.561, 973791.631 7...",f,temmelig eksponert,moderately exposed


In [ ]:
gdf.to_file(FIN, driver="GPKG")
#153188 -> 152999

In [ ]:
AOI = "gs://niva-geodata/MarintNaturKart/aux/aoi_from_marine_vanntyper.geo.parquet"

waves.classify.clip_to_aoi(VRAW, VCLIP, AOI)
